In [ ]:
import re
import glob
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_style("dark")

rows = []

# Parse data from logs into a DataFrame
for filename in glob.glob("mma-accuracy-*.log"):
    match = re.match(r"mma-accuracy-(sm\d+[a]?)_([a-z0-9_]+)\.log", filename)
    if not match:
        continue
    arch, prec = match.groups()
    if prec == 'fp32':
        continue

    with open(filename) as f:
        for line in f:
            m = re.match(r"\[\s*([^]]+)\s*\]\s*\[\s*([^]]+)\s*\]\s*error:\s*([^\s]+)\s+([^\s]+)", line)
            if m:
                a_vals = [v.strip() for v in m.group(1).split(",")]
                b_vals = [v.strip() for v in m.group(2).split(",")]
                abs_err = m.group(3)
                rel_err = m.group(4)

                a_min, a_max = a_vals
                b_min, b_max = b_vals
                
                rows.append((arch, prec, a_min, a_max, b_min, b_max, abs_err, rel_err))

df = pd.DataFrame(rows, columns=["arch", "prec", "a_min", "a_max", "b_min", "b_max", "abs_err", "rel_err"])

numeric_cols = ["a_min", "a_max", "b_min", "b_max", "abs_err", "rel_err"]
df[numeric_cols] = df[numeric_cols].astype(float)

# Scatterplot with seaborn
plt.figure(figsize=(16,8))
sns.scatterplot(
    data=df,
    x="a_max", y="rel_err",
    hue="prec",       # different colors per GPU architecture
    style="arch",     # different markers per precision
    s=50, alpha=0.8
)

plt.xscale("log")
plt.yscale("log")
plt.xlabel("Max value")
plt.ylabel("Relative error")
plt.title("MMA accuracy per architecture and precision")
plt.grid(True, which="both", ls="--", alpha=0.8)

ax = plt.gca()

# ax.set_xlim([1e-6, 1e5])
# ax.set_ylim([1e-3, 3])

epsilon_values = {
    'fp16': 2**(-10), # 10 mantissa bits
    'bf16': 2**(-7),  # 7 mantissa bits
    'e4m3': 2**(-3),  # 3 mantissa bits
    'e5m2': 2**(-2),  # 2 mantissa bits
}

mantissa_bits = {
    'fp16': 10,
    'bf16': 7,
    'e4m3': 3,
    'e5m2': 2,
}

formats = ['fp16', 'bf16', 'e4m3', 'e5m2']
prop_cycle = plt.rcParams['axes.prop_cycle']
colors = prop_cycle.by_key()['color']

for fmt, color in zip(formats, colors):
    plt.axhline(y=epsilon_values[fmt], color=color, linestyle=':', alpha=0.7, lw=3,
                label=f'{fmt} ε = {epsilon_values[fmt]:.3e}')

    plt.text(0.99, 1.2*epsilon_values[fmt], f'mantissa bits: {mantissa_bits[fmt]}', color=color,
         transform=ax.get_yaxis_transform(),  # Uses data coordinates for y, axes for x
         verticalalignment='bottom', horizontalalignment='right', fontsize=12,
         bbox=dict(boxstyle="round,pad=0.3", facecolor='white', alpha=0.8, edgecolor=None))
    
plt.tight_layout()
plt.show()


In [ ]:
total_width = 20
ncols = df["arch"].nunique()
height = 8
aspect = (total_width / ncols) / height

g = sns.FacetGrid(df, col="arch", sharey=True, sharex=True, height=height, aspect=aspect)
g.map_dataframe(
    sns.scatterplot,
    x="a_max", y="rel_err",
    hue="prec", style="prec",
    s=50,
    alpha=0.8
)
g.set(xscale="log", yscale="log")
g.add_legend()
    
for ax in g.axes.flatten():
    ax.set_xlabel("Max value")
    ax.set_ylabel("Relative error")
    ax.grid(True, which="both", ls="--", alpha=0.8)

In [ ]:
total_width = 20
ncols = df["arch"].nunique()
height = 8
aspect = (total_width / ncols) / height

g = sns.FacetGrid(df, col="prec", sharey=True, sharex=True, height=height, aspect=aspect)
g.map_dataframe(
    sns.scatterplot,
    x="a_max", y="rel_err",
    hue="arch", style="arch",
    s=50,
    alpha=0.8
)
g.set(xscale="log", yscale="log")
g.add_legend()
    
for ax in g.axes.flatten():
    ax.set_xlabel("Max value")
    ax.set_ylabel("Relative error")
    ax.grid(True, which="both", ls="--", alpha=0.8)

In [ ]:
targets = [1e-1, 1e-2, 1e-3]

results = []

for target in targets:
    for (prec, arch), g in df.groupby(["prec", "arch"]):
        # Only consider rows where error <= target
        g_valid = g[g["rel_err"] <= target]
        if g_valid.empty:
            continue

        # Pick the row with the largest a_max that still meets the target
        row = g_valid.loc[g_valid["a_max"].idxmax()]
        results.append({
            "target_error": target,
            "prec": prec,
            "arch": arch,
            "a_max": row["a_max"],
            "actual_error": row["rel_err"]
        })

resdf = pd.DataFrame(results)

# Pivot table to have architectures as columns
# Combine a_max and actual_error into a string for display
resdf["display"] = resdf.apply(lambda r: f"{r['a_max']:.5g} ({r['actual_error']:.2g})", axis=1)

pivot = resdf.pivot_table(
    index=["prec", "target_error"],
    columns="arch",
    values="display",
    aggfunc="first"
)

# Highlight the architecture with the largest a_max
# We need a numeric pivot for highlighting
numeric_pivot = resdf.pivot_table(
    index=["prec", "target_error"],
    columns="arch",
    values="a_max",
    aggfunc="first"
)

styled = pivot.style.apply(lambda df_: numeric_pivot.eq(numeric_pivot.max(axis=1), axis=0)
                          .replace({True: "background-color: lightgreen", False: ""}),
                          axis=None)

styled

In [ ]:
import matplotlib.pyplot as plt
import itertools

# Define markers to cycle through
markers = ["o", "s", "D", "^", "v", "P", "*", "X", "h"]
marker_cycle = itertools.cycle(markers)

# -----------------------------
# Facet-like plot by architecture
# -----------------------------
arches_unique = df["arch"].unique()
precs_unique = df["prec"].unique()
fig, axes = plt.subplots(1, len(arches_unique), figsize=(16,5), sharey=True)

for ax, arch in zip(axes, arches_unique):
    df_arch = df[df["arch"] == arch]
    # create a fresh marker cycle per facet to keep styles consistent
    marker_cycle = itertools.cycle(markers)
    for prec in precs_unique:
        df_sub = df_arch[df_arch["prec"] == prec]
        ax.scatter(df_sub["a_max"], df_sub["rel_err"],
                   label=prec,
                   s=50, alpha=0.6,
                   marker=next(marker_cycle))
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_title(f"{arch}")
    ax.set_xlabel("Max value")

axes[0].set_ylabel("Relative Error")
axes[0].legend(title="Precision")
plt.tight_layout()
plt.show()

# -----------------------------
# Facet-like plot by precision
# -----------------------------
fig, axes = plt.subplots(1, len(precs_unique), figsize=(16,5), sharey=True)

for ax, prec in zip(axes, precs_unique):
    df_prec = df[df["prec"] == prec]
    marker_cycle = itertools.cycle(markers)
    for arch in arches_unique:
        df_sub = df_prec[df_prec["arch"] == arch]
        ax.scatter(df_sub["a_max"], df_sub["rel_err"],
                   label=arch,
                   s=50, alpha=0.6,
                   marker=next(marker_cycle))
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_title(f"{prec}")
    ax.set_xlabel("Max value")

axes[0].set_ylabel("Relative Error")
axes[0].legend(title="Architecture")
plt.tight_layout()
plt.show()